# Voyexa AI — Model 3: Content-based Similarity Recommender (core ranking engine)
Given a user's chosen interests + suitable_for profile (or a seed place they liked), rank candidates by cosine similarity of their tag/attribute vectors.

This is the ML half of the hybrid system: **RULES filter** candidates (budget, difficulty/age safety, season, state) → **this MODEL ranks** what's left.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = os.path.join("..", "..", "data", "processed", "places_processed.csv")
ARTIFACTS_DIR = os.path.join("..", "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape

(1163, 202)

## Build the per-place feature vector
Tags + suitable_for as binary flags, plus a light (down-weighted) nudge from difficulty and cost tier so similarity is mostly about interests/suitability.

In [2]:
interest_cols = [c for c in df.columns if c.startswith("interest_")]
suitable_cols = [c for c in df.columns if c.startswith("suitable_")]
vec_cols = interest_cols + suitable_cols

extra_scaled = df[["difficulty_score", "total_daily_cost_avg"]].copy()
extra_scaled = pd.DataFrame(
    MinMaxScaler().fit_transform(extra_scaled) * 0.3,
    columns=["difficulty_norm", "cost_norm"], index=df.index
)

feature_matrix = pd.concat([df[vec_cols].astype(float), extra_scaled], axis=1)
similarity_matrix = cosine_similarity(feature_matrix)

print(f"Feature vector size per place: {feature_matrix.shape[1]}")
print(f"Similarity matrix shape: {similarity_matrix.shape}")

Feature vector size per place: 168
Similarity matrix shape: (1163, 1163)


## Function 1: recommend places similar to a seed place

In [3]:
def recommend_similar(place_id, top_n=5):
    idx = df.index[df["place_id"] == place_id][0]
    scores = similarity_matrix[idx]
    top_idx = np.argsort(scores)[::-1]
    top_idx = [i for i in top_idx if i != idx][:top_n]
    return df.loc[top_idx, ["place_id", "Place_Name", "City", "State", "Interest_Tags"]].assign(
        similarity=scores[top_idx].round(3)
    )

sample_id = df["place_id"].iloc[0]
print(f"Seed place: {df.loc[df['place_id']==sample_id, 'Place_Name'].values[0]}")
recommend_similar(sample_id, top_n=5)

Seed place: Dal Lake


,place_id,Place_Name,City,State,Interest_Tags,similarity
7,8,Wular Lake,Bandipora,Jammu And Kashmir,"Scenic,Nature,Birdwatching",0.833
6,7,Manasbal,Ganderbal,Jammu And Kashmir,"Nature,Birdwatching,Scenic",0.833
67,70,Kausani,Kausani,Uttarakhand,"Scenic,Nature,Romantic",0.832
484,487,Nelliampathy,Palakkad,Kerala,"Nature,Scenic,Relaxation",0.832
9,10,Pahalgam,Anantnag,Jammu And Kashmir,"Scenic,Nature,Adventure",0.832


## Function 2: recommend from a user's interest profile
`candidate_place_ids` should be the RULE-FILTERED shortlist (budget/difficulty/season already applied upstream) — pass `None` to rank the whole dataset for testing.

In [4]:
def recommend_from_profile(interest_tags, suitable_for, candidate_place_ids=None, top_n=10):
    query = pd.Series(0.0, index=vec_cols)
    for tag in interest_tags:
        col = f"interest_{tag.lower().replace(' ', '_')}"
        if col in query.index:
            query[col] = 1.0
    for s in suitable_for:
        col = f"suitable_{s.lower().replace(' ', '_')}"
        if col in query.index:
            query[col] = 1.0
    query_full = pd.concat([query, pd.Series({"difficulty_norm": 0.0, "cost_norm": 0.0})])

    candidates = df if candidate_place_ids is None else df[df["place_id"].isin(candidate_place_ids)]
    cand_matrix = feature_matrix.loc[candidates.index]

    sims = cosine_similarity([query_full.values], cand_matrix.values)[0]
    ranked_idx = np.argsort(sims)[::-1][:top_n]
    result_idx = candidates.index[ranked_idx]

    return df.loc[result_idx, ["place_id", "Place_Name", "City", "State", "Interest_Tags", "Popularity_Rating"]].assign(
        similarity=sims[ranked_idx].round(3)
    )

recommend_from_profile(interest_tags=["Adventure", "Trekking"], suitable_for=["Friends"], top_n=5)

,place_id,Place_Name,City,State,Interest_Tags,Popularity_Rating,similarity
1020,1023,Prabalgad Fort,Panvel,Maharashtra,"Trekking,Adventure",4.6,0.864
986,989,Harishchandragad,Ahmednagar,Maharashtra,"Trekking,Adventure",4.8,0.863
987,990,Kalavantin Durg,Panvel,Maharashtra,"Trekking,Adventure",4.7,0.863
824,827,Roopkund Lake,Chamoli,Uttarakhand,"Trekking,Adventure",4.7,0.863
407,410,Mount Saramati,Kiphire,Nagaland,"Adventure,Trekking",4.6,0.856


## Save artifacts for the backend to reuse at inference time

In [5]:
joblib.dump(feature_matrix, os.path.join(ARTIFACTS_DIR, "similarity_feature_matrix.pkl"))
joblib.dump(vec_cols, os.path.join(ARTIFACTS_DIR, "similarity_vec_cols.pkl"))
print("Saved feature matrix + vec_cols to ml/artifacts/")

Saved feature matrix + vec_cols to ml/artifacts/
